# **01 · Preprocesamiento de datos y generación de features**

**Proyecto:** detector de limones con YOLO para un robot recolector (ver `README.md`).

Este notebook convierte los datos en bruto en un dataset listo para entrenar:

1. Descarga los dos datasets desde Roboflow Universe.
2. Los limpia y los unifica en una sola clase, `limon`.
3. Genera **cuatro features nuevas** por cada limón etiquetado (tamaño, forma, color y densidad).
4. Guarda el dataset unificado en `data/datos_limon.zip` y las tablas en `results/`.

> **Antes de empezar (Colab):** guardar la API key de [roboflow.com](https://roboflow.com)
> (*Settings → API Keys*) como secreto de Colab con el nombre `ROBOFLOW_API_KEY` (ícono de la llave,
> a la izquierda) y habilitar el acceso para este cuaderno. Este notebook no necesita GPU.

## **1. Preparación del entorno**

In [ ]:
# Instalamos el cliente de Roboflow (OpenCV, Pandas y Matplotlib ya vienen en Colab).
!pip -q install roboflow

In [ ]:
# Carpetas del proyecto. En Colab todo queda en Drive (carpeta limon_ia), con la misma estructura
# que el repositorio: data/, results/ y runs/ (modelos entrenados). Fuera de Colab se usa el repositorio.
from pathlib import Path

try:
    from google.colab import drive
    drive.mount("/content/drive")
    BASE = Path("/content/drive/MyDrive/limon_ia")   # carpeta del proyecto en Drive
    TRABAJO = Path("/content")                        # disco de Colab: rápido, pero se borra al cerrar
except ImportError:
    BASE = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
    TRABAJO = BASE / "data"

DATA, RESULTS, RUNS = BASE / "data", BASE / "results", BASE / "runs"
for carpeta in (DATA, RESULTS, RUNS):
    carpeta.mkdir(parents=True, exist_ok=True)
DATOS = TRABAJO / "datos"                             # dataset unificado, descomprimido
print("Proyecto en", BASE)

In [ ]:
# Biblioteca estándar: variables de entorno, expresiones regulares, copia de archivos y azar.
import os
import re
import shutil
import random

# Imágenes, cálculo, tablas, gráficos y archivos de configuración.
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import yaml

## **2. Descarga de los datos en bruto**

| Dataset | Qué es | Para qué |
|---|---|---|
| Base: [brinjalproject/lemons-segmentation](https://universe.roboflow.com/brinjalproject/lemons-segmentation) (v5) | Dataset público de limones en el árbol, etiquetado por terceros como segmentación (polígonos), con 3 clases | Dar al modelo muchos ejemplos de "limón entre hojas" |
| Propio: [hectors-workspace-e5dhh/limones-amarillos](https://universe.roboflow.com/hectors-workspace-e5dhh/limones-amarillos) (v3) | 50 fotos del limonero en maceta del laboratorio, sacadas con celular y etiquetadas por el Grupo 2 de Robótica (30 train / 10 valid / 10 test) | Adaptar el modelo al escenario real y evaluarlo |

Si un dataset ya está bajado, no se vuelve a descargar.

In [ ]:
# (workspace, proyecto, versión, carpeta destino), tal como aparecen en la URL de Roboflow Universe.
BASE_RF = ("brinjalproject", "lemons-segmentation", 5, TRABAJO / "base_roboflow")
PROPIO_RF = ("hectors-workspace-e5dhh", "limones-amarillos", 3, TRABAJO / "limones_roboflow")


# --------
# Función que baja una versión de un proyecto de Roboflow en formato YOLO.
#   - Deja las carpetas train/, valid/ y test/ (cada una con images/ y labels/) y un data.yaml.
# --------
def bajar(workspace, proyecto, version, destino):
    if (destino / "data.yaml").exists():
        print(f"{proyecto}: ya estaba bajado en {destino}")
        return
    from roboflow import Roboflow
    try:
        from google.colab import userdata
        clave = userdata.get("ROBOFLOW_API_KEY")
    except ImportError:
        clave = os.environ["ROBOFLOW_API_KEY"]
    rf = Roboflow(api_key=clave)
    rf.workspace(workspace).project(proyecto).version(version).download("yolov8", location=str(destino),
                                                                         overwrite=True)


bajar(*BASE_RF)
bajar(*PROPIO_RF)

## **3. Limpieza y unificación**

Cada imagen `N.jpg` tiene un archivo `N.txt` con una línea por objeto: `clase x_centro y_centro ancho alto`,
con las coordenadas **normalizadas** entre 0 y 1. Los datos en bruto no se pueden usar tal cual:

| Problema en los datos en bruto | Qué se hace |
|---|---|
| El dataset base está etiquetado con **polígonos** (segmentación) | Cada polígono se reemplaza por la caja que lo encierra |
| El dataset base tiene 3 clases (amarillo, verde, enfermo) y el robot solo cosecha amarillos | Solo la clase amarilla pasa a ser `limon` (clase 0) |
| Hay imágenes con limones verdes o enfermos | **Se descartan enteras**: si quedaran con esos limones sin etiquetar, el modelo aprendería que un limón es fondo |
| Cada dataset numera sus clases distinto | Todo se renumera a una sola clase, `0 = limon` |

**Revisar lo que imprime la celda:** la fila marcada `-> limon` tiene que ser la de limones amarillos.

In [ ]:
BASE_AMARILLO = r"yellow|amarill"     # clase del dataset base que se usa como "limon"
OTRO_LIMON = r"lemon|lim[oó]n"        # otras clases que también son limones: sus imágenes se descartan


# --------
# Función que convierte una etiqueta en caja.
#   - Recibe: los números de una línea sin la clase (4 de una caja, o 2 por vértice de un polígono).
#   - Devuelve: [x_centro, y_centro, ancho, alto] como texto.
# --------
def a_caja(campos):
    if len(campos) == 4:
        return campos
    xs, ys = [float(c) for c in campos[0::2]], [float(c) for c in campos[1::2]]
    return [f"{(min(xs) + max(xs)) / 2:.6f}", f"{(min(ys) + max(ys)) / 2:.6f}",
            f"{max(xs) - min(xs):.6f}", f"{max(ys) - min(ys):.6f}"]


# --------
# Función que copia un conjunto (train, valid o test) de un dataset con la clase de limón en 0.
#   - Recibe: carpeta de origen, conjunto, carpeta destino y la expresión regular de la clase "limon".
#   - Devuelve: [imágenes copiadas, limones, imágenes descartadas].
# --------
def importar(origen, conjunto, destino, es_limon):
    nombres = yaml.safe_load((origen / "data.yaml").read_text())["names"]
    if isinstance(nombres, dict):
        nombres = [nombres[k] for k in sorted(nombres)]
    (destino / "images").mkdir(parents=True, exist_ok=True)
    (destino / "labels").mkdir(parents=True, exist_ok=True)
    copiadas = limones = descartadas = 0
    carpeta = origen / conjunto / "images"
    for img in sorted(carpeta.iterdir()) if carpeta.exists() else []:
        txt = origen / conjunto / "labels" / (img.stem + ".txt")
        lineas, descartar = [], False
        for linea in txt.read_text().splitlines() if txt.exists() else []:
            if not linea.strip():
                continue
            clase = nombres[int(linea.split()[0])]
            if re.search(es_limon, clase, re.I):
                lineas.append(" ".join(["0"] + a_caja(linea.split()[1:])))
            elif re.search(OTRO_LIMON, clase, re.I):
                descartar = True          # limón de otra clase: la imagen entera se descarta
        if descartar:
            descartadas += 1
            continue
        shutil.copy(img, destino / "images" / img.name)
        (destino / "labels" / (img.stem + ".txt")).write_text("\n".join(lineas) + ("\n" if lineas else ""))
        copiadas += 1
        limones += len(lineas)
    return [copiadas, limones, descartadas]


# Mostramos qué se hace con cada clase del dataset base.
nombres = yaml.safe_load((BASE_RF[3] / "data.yaml").read_text())["names"]
for n in (nombres.values() if isinstance(nombres, dict) else nombres):
    print(f"clase '{n}':", "-> limon" if re.search(BASE_AMARILLO, n, re.I) else
          "se descartan sus imágenes" if re.search(OTRO_LIMON, n, re.I) else "se borra (queda como fondo)")

# Borramos lo armado en una ejecución anterior y copiamos los dos datasets.
shutil.rmtree(DATOS, ignore_errors=True)
filas = []
for conjunto in ("train", "valid", "test"):
    filas.append(["base", conjunto] + importar(BASE_RF[3], conjunto, DATOS / "base" / conjunto, BASE_AMARILLO))
for conjunto in ("train", "valid", "test"):
    filas.append(["propio", conjunto] + importar(PROPIO_RF[3], conjunto, DATOS / "propio" / conjunto, "."))

estadisticas = pd.DataFrame(filas, columns=["dataset", "conjunto", "imágenes", "limones", "imágenes descartadas"])
estadisticas.to_csv(RESULTS / "estadisticas_dataset.csv", index=False)
estadisticas

**Cómo se usan los conjuntos.** Del dataset base, los tres conjuntos se usan para **entrenar**. Del dataset
propio: `train` (30 fotos) para entrenar, `valid` (10) para elegir el mejor modelo y los hiperparámetros, y
`test` (10) queda sin tocar hasta la evaluación final del notebook 04.

### **3.1 Muestras con sus etiquetas**

Antes de entrenar conviene comprobar que las cajas caen sobre limones. Arriba, el dataset base; abajo, las
fotos propias.

In [ ]:
# --------
# Función que dibuja las cajas de las etiquetas sobre una imagen.
#   - Recibe: ruta de la imagen (su .txt está en la carpeta labels/ hermana de images/).
#   - Devuelve: la imagen en RGB con las cajas dibujadas.
# --------
def dibujar_etiquetas(img):
    im = cv2.cvtColor(cv2.imread(str(img)), cv2.COLOR_BGR2RGB)
    alto, ancho = im.shape[:2]
    for linea in (img.parent.parent / "labels" / (img.stem + ".txt")).read_text().splitlines():
        if not linea.strip():
            continue
        # Desnormalizamos: de fracciones (0 a 1) a píxeles, y de (centro, ancho, alto) a esquinas.
        _, xc, yc, w, h = map(float, linea.split()[:5])
        cv2.rectangle(im, (int((xc - w / 2) * ancho), int((yc - h / 2) * alto)),
                      (int((xc + w / 2) * ancho), int((yc + h / 2) * alto)), (0, 90, 255), max(2, ancho // 300))
    return im


azar = random.Random(0)
muestras = (azar.sample(sorted((DATOS / "base" / "train" / "images").iterdir()), 3)
            + azar.sample(sorted((DATOS / "propio" / "train" / "images").iterdir()), 3))
plt.figure(figsize=(15, 9))
for i, img in enumerate(muestras):
    plt.subplot(2, 3, i + 1)
    plt.imshow(dibujar_etiquetas(img))
    plt.title(("Base: " if i < 3 else "Propio: ") + img.name[:25])
    plt.axis("off")
plt.tight_layout()
plt.savefig(RESULTS / "fig_muestras_dataset.jpg", dpi=80)
plt.show()

## **4. Generación de features**

A partir de los datos crudos (los píxeles de la imagen y los cuatro números de cada caja) se calculan
cuatro características nuevas por cada limón etiquetado:

| Feature | Cómo se genera | Qué información aporta |
|---|---|---|
| **Tamaño** (`lado_px`, `tamano`) | Raíz del área de la caja, llevada a una imagen de 640 px: $\sqrt{w \cdot h}\cdot 640$. Se agrupa en chico (< 32 px), mediano (32 a 96 px) y grande (> 96 px), los cortes que usa COCO | Equivale a la distancia: un limón lejano se ve chico. Los objetos chicos son los más difíciles de detectar |
| **Forma** (`aspecto`) | Ancho de la caja dividido por su alto, en píxeles: $(w \cdot W)/(h \cdot H)$ | Un limón entero da un valor cercano a 1. Si se aleja mucho, el limón está tapado por hojas o cortado por el borde |
| **Color** (`tono`, `saturacion`, `brillo`) | Se pasa la imagen a HSV y se toma la mediana de cada canal en la mitad central de la caja (para no mezclar el fondo). El tono se expresa en grados: amarillo ≈ 50°–65°, verde ≈ 70°–140° | Madurez e iluminación: un limón a la sombra o verdoso se parece más a una hoja |
| **Densidad** (`n_limones`) | Cantidad de limones etiquetados en la misma imagen | En los racimos los limones se tapan entre sí y aparecen detecciones duplicadas |

**Para qué se usan.** YOLO recibe la imagen completa y aprende sus propias características internas, así que
estas features no entran como columnas al modelo. Se usan para: (1) comparar el dataset público con las fotos
propias y ver si se parecen, (2) justificar los hiperparámetros de aumento de datos que se optimizan en el
notebook 03 (`hsv_h` varía el tono; `scale`, el tamaño) y (3) analizar en el notebook 04 en qué tipo de limón
se equivoca el modelo.

In [ ]:
LADO_REF = 640     # tamaño de imagen de referencia para medir el lado del limón en píxeles


# --------
# Función que calcula las features de todos los limones de una imagen.
#   - Recibe: ruta de la imagen, nombre del dataset y del conjunto.
#   - Devuelve: una lista de diccionarios, uno por limón etiquetado.
# --------
def features_de_imagen(img, dataset, conjunto):
    lineas = [l.split() for l in (img.parent.parent / "labels" / (img.stem + ".txt")).read_text().splitlines()
              if l.strip()]
    if not lineas:
        return []
    bgr = cv2.imread(str(img))
    alto, ancho = bgr.shape[:2]
    hsv = cv2.cvtColor(bgr, cv2.COLOR_BGR2HSV)
    filas = []
    for n, campos in enumerate(lineas):
        xc, yc, w, h = map(float, campos[1:5])
        # Mitad central de la caja, en píxeles (al menos 1 píxel).
        x0, x1 = int((xc - w / 4) * ancho), max(int((xc + w / 4) * ancho), int((xc - w / 4) * ancho) + 1)
        y0, y1 = int((yc - h / 4) * alto), max(int((yc + h / 4) * alto), int((yc - h / 4) * alto) + 1)
        centro = hsv[max(0, y0):y1, max(0, x0):x1].reshape(-1, 3)
        lado = (w * h) ** 0.5 * LADO_REF
        filas.append({
            "dataset": dataset, "conjunto": conjunto, "imagen": img.name, "n": n,
            "xc": xc, "yc": yc, "w": w, "h": h,
            "lado_px": round(lado, 1),
            "tamano": "chico" if lado < 32 else "mediano" if lado <= 96 else "grande",
            "aspecto": round((w * ancho) / (h * alto), 3),
            "tono": float(np.median(centro[:, 0])) * 2,          # OpenCV guarda el tono de 0 a 179: x2 = grados
            "saturacion": round(float(np.median(centro[:, 1])) / 255, 3),
            "brillo": round(float(np.median(centro[:, 2])) / 255, 3),
            "n_limones": len(lineas),
        })
    return filas


filas = []
for dataset in ("base", "propio"):
    for conjunto in ("train", "valid", "test"):
        carpeta = DATOS / dataset / conjunto / "images"
        for img in sorted(carpeta.iterdir()) if carpeta.exists() else []:
            filas += features_de_imagen(img, dataset, conjunto)
features = pd.DataFrame(filas)
features.to_csv(RESULTS / "features_limones.csv", index=False)
print(len(features), "limones con features ->", RESULTS / "features_limones.csv")
features.head()

### **4.1 Dataset público contra fotos propias**

Si las features de los dos datasets se distribuyen distinto, el modelo aprende de imágenes que no se parecen a
las que va a ver el robot. Esa diferencia es la que se intenta compensar repitiendo las fotos propias
(sobremuestreo) y con el aumento de datos.

In [ ]:
resumen = features.groupby("dataset")[["lado_px", "aspecto", "tono", "saturacion", "brillo", "n_limones"]] \
    .median().round(2)
resumen.to_csv(RESULTS / "features_resumen.csv")
print("Mediana de cada feature por dataset:")
display(resumen)
print("Limones por tamaño (%):")
display((pd.crosstab(features["dataset"], features["tamano"], normalize="index") * 100).round(1))

# Un histograma por feature, con los dos datasets superpuestos (normalizados para poder compararlos).
columnas = [("lado_px", "Tamaño: lado del limón [px, imagen de 640]"), ("aspecto", "Forma: ancho / alto"),
            ("tono", "Color: tono [grados]"), ("n_limones", "Densidad: limones en la imagen")]
fig, ejes = plt.subplots(1, 4, figsize=(20, 4))
for ax, (col, titulo) in zip(ejes, columnas):
    rango = (features[col].quantile(0.01), features[col].quantile(0.99))
    for dataset, color in (("base", "#1f77b4"), ("propio", "#ff7f0e")):
        ax.hist(features.loc[features["dataset"] == dataset, col], bins=25, range=rango, density=True,
                alpha=0.55, color=color, label=dataset)
    ax.set_title(titulo)
    ax.legend()
plt.tight_layout()
plt.savefig(RESULTS / "fig_features.png", dpi=110)
plt.show()

**Análisis (completar después de correr).** Mirar la tabla de medianas y los histogramas y anotar acá en qué
se diferencian las fotos propias del dataset público: tamaño de los limones (¿más chicos, por estar sacadas
desde más lejos?), tono y cantidad de limones por imagen.

## **5. Guardar el dataset procesado**

El dataset unificado se guarda en un solo archivo, `data/datos_limon.zip`. Los notebooks 02, 03 y 04 lo
descomprimen y trabajan sobre él, sin volver a descargar ni a limpiar.

In [ ]:
shutil.make_archive(str(TRABAJO / "datos_limon"), "zip", DATOS)
if TRABAJO != DATA:
    shutil.copy(TRABAJO / "datos_limon.zip", DATA / "datos_limon.zip")
print(f"Guardado: {DATA / 'datos_limon.zip'} ({(DATA / 'datos_limon.zip').stat().st_size / 1e6:.0f} MB)")